
we are  workring  for a ecommerce company called Gizxmo Box , snd they want to create data lake house using apache spark ; 

sell electronics equipments direct to consumer : 

and has operatinal data : 

 and extrnal data 

In [0]:
%python
VOLUME_PATH = "/Volumes/ud_cata/gizmobox/raw"

# See all files in raw volume
display(dbutils.fs.ls(VOLUME_PATH))

In [0]:
%python
import zipfile, os

VOLUME_PATH = "/Volumes/ud_cata/gizmobox/raw"
ZIP_NAME = "gizmobox_data.zip"   # ← replace with exact name from Cell 1

with zipfile.ZipFile(f"{VOLUME_PATH}/{ZIP_NAME}", "r") as z:
    z.extractall(f"{VOLUME_PATH}/landing/")

print("✅ Unzipped!")

In [0]:
%python
for root, dirs, files in os.walk(f"{VOLUME_PATH}/landing/"):
    level = root.replace(f"{VOLUME_PATH}/landing/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:3]:  # show first 3 files per folder
        print(f"{'  ' * (level+1)}{f}")

In [0]:
%python
# Base path — all real data is inside gizmobox/ folder
BASE = "/Volumes/ud_cata/gizmobox/raw/landing/gizmobox"

# Define all source paths
CUSTOMERS_PATH    = f"{BASE}/customers/"
ADDRESSES_PATH    = f"{BASE}/addresses/"
ORDERS_PATH       = f"{BASE}/orders/"
MEMBERSHIPS_PATH  = f"{BASE}/memberships/"
PAYMENTS_PATH     = f"{BASE}/payments/"
CUSTOMERS_AL_PATH = f"{BASE}/customers_autoloader/"   # for autoloader section later
CUSTOMERS_ST_PATH = f"{BASE}/customers_stream/"       # for streaming section later

print("✅ Paths set!")

In [0]:
%python
# CUSTOMERS — JSON
customers_df = spark.read.json(CUSTOMERS_PATH)
print(f"Customers: {customers_df.count()} rows")
display(customers_df.printSchema())

In [0]:
%python
# ── Cell 5 — Read & Verify All Files ──────────────────────────

# CUSTOMERS — JSON
customers_df = spark.read.json(CUSTOMERS_PATH)
print(f"✅ Customers: {customers_df.count()} rows")
display(customers_df)

In [0]:
%python
# ADDRESSES — TSV
addresses_df = spark.read \
    .option("header", True) \
    .option("sep", "\t") \
    .csv(ADDRESSES_PATH)
print(f"✅ Addresses: {addresses_df.count()} rows")
display(addresses_df)

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
USE CATALOG gizmobox;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gizmobox;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gizmobox.landing;
CREATE SCHEMA IF NOT EXISTS gizmobox.bronze;
CREATE SCHEMA IF NOT EXISTS gizmobox.silver;
CREATE SCHEMA IF NOT EXISTS gizmobox.gold;

In [0]:
%sql
SHOW SCHEMAS IN gizmobox;

In [0]:
%sql
DROP SCHEMA IF EXISTS ud_cata.gizmobox_bronze CASCADE;
DROP SCHEMA IF EXISTS ud_cata.gizmobox_silver CASCADE;
DROP SCHEMA IF EXISTS ud_cata.gizmobox_gold CASCADE;
DROP SCHEMA IF EXISTS ud_cata.gizmobox CASCADE;

In [0]:
%python

# Check if volume data is still there
dbutils.fs.ls("/Volumes/ud_cata/gizmobox/raw/landing/gizmobox/")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS gizmobox.landing.raw;


In [0]:
# ── Check volume first ─────────────────────────────────────────
dbutils.fs.ls("/Volumes/gizmobox/landing/raw/")


In [0]:
# ── Unzip ──────────────────────────────────────────────────────
import zipfile, os

VOLUME_PATH = "/Volumes/gizmobox/landing/raw"

# Check exact zip filename first
files = dbutils.fs.ls(VOLUME_PATH)
for f in files:
    print(f.name)

In [0]:
# ── Replace zip name with what you see above ───────────────────
ZIP_NAME = "gizmobox-data.zip"   # ← confirm this matches

with zipfile.ZipFile(f"{VOLUME_PATH}/{ZIP_NAME}", "r") as z:
    z.extractall(f"{VOLUME_PATH}/landing/")

print("✅ Unzipped!")

In [0]:
# See exact filename
for f in dbutils.fs.ls("/Volumes/gizmobox/landing/raw/"):
    print(f.name)

In [0]:
import zipfile, os

VOLUME_PATH = "/Volumes/gizmobox/landing/raw"
ZIP_NAME = "gizmobox_data.zip"   # ← underscore not hyphen

with zipfile.ZipFile(f"{VOLUME_PATH}/{ZIP_NAME}", "r") as z:
    z.extractall(f"{VOLUME_PATH}/landing/")

print("✅ Unzipped!")

In [0]:
for root, dirs, files in os.walk(f"{VOLUME_PATH}/landing/"):
    level = root.replace(f"{VOLUME_PATH}/landing/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:2]:
        print(f"{'  ' * (level+1)}{f}")

In [0]:
# ── Source (where files are now) ──────────────────────────────
BASE = "/Volumes/gizmobox/landing/raw/landing/gizmobox"

# ── Destinations (new volumes) ────────────────────────────────
OPERATIONAL = "/Volumes/gizmobox/landing/operational"
EXTERNAL    = "/Volumes/gizmobox/landing/external"

# Copy operational data files
dbutils.fs.cp(f"{BASE}/customers/",   f"{OPERATIONAL}/customers/",   recurse=True)
dbutils.fs.cp(f"{B/Volumes/gizmobox/landing/operational/customers/customers_2024_10.jsonASE}/addresses/",   f"{OPERATIONAL}/addresses/",   recurse=True)
dbutils.fs.cp(f"{BASE}/orders/",      f"{OPERATIONAL}/orders/",      recurse=True)
dbutils.fs.cp(f"{BASE}/memberships/", f"{OPERATIONAL}/memberships/", recurse=True)
print("✅ Operational data copied!")

# # Copy external data files
# dbutils.fs.cp(f"{BASE}/payments/",    f"{EXTERNAL}/payments/",       recurse=True)
# print("✅ External data copied!")

# # Copy autoloader & stream folders (for later sections)
# dbutils.fs.cp(f"{BASE}/customers_autoloader/", f"{OPERATIONAL}/customers_autoloader/", recurse=True)
# dbutils.fs.cp(f"{BASE}/customers_stream/",     f"{OPERATIONAL}/customers_stream/",     recurse=True)
# print("✅ Autoloader & Stream data copied!")

In [0]:
%sql
    -- # /Volumes/gizmobox/landing/operational/customers/customers_2024_10.json
-- querying single file 
    SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/customers_2024_10.json`

In [0]:
%sql
    -- # /Volumes/gizmobox/landing/operational/customers/customers_2024_10.json
-- querying MUltiple file 
    SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/customers_2024_*.json`

In [0]:
%sql
    -- # /Volumes/gizmobox/landing/operational/customers/customers_2024_10.json
-- querying folder   
    SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers`


## Creating View 


In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_customers 
AS 
SELECT *,
    _metadata.file_path AS file_path
  FROM json.`/Volumes/gizmobox/landing/operational/customers`
      

In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_customers


##Creating Temporary view

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_customers
AS 
SELECT *,
    _metadata.file_path AS file_path
  FROM json.`/Volumes/gizmobox/landing/operational/customers`

In [0]:
%sql

SELECT * FROM tv_customers;

## creating gloabal temp view 

In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMPORARY VIEW gtv_customers
AS 
SELECT *,
    _metadata.file_path AS file_path
  FROM json.`/Volumes/gizmobox/landing/operational/customers`

In [0]:
# %sql
# SELECT * FROM global_temp.gtv_customers


##Extract data from the orders json file

- 1, query orders file using json format
- 2, query orders filke using text format 
- 3, create ordes view in bronze schema 



#### 1, query orders file using json format

In [0]:
%sql 

SELECT * FROM json.`/Volumes/gizmobox/landing/operational/orders`


### 2, query orders filke using text form



In [0]:
%sql 

SELECT * FROM text.`/Volumes/gizmobox/landing/operational/orders`


## Creating Orders view in bronze schema 

In [0]:
%sql 
CREATE OR REPLACE VIEW gizmobox.bronze.v_orders
AS
SELECT * FROM text.`/Volumes/gizmobox/landing/operational/orders`

In [0]:
%sql 

SELECT * FROM gizmobox.bronze.v_orders


## Now  in this lesson we are going to extraxcrt membershipo data with binary files

###Extract data with the membership folder using binary file format. 

- 1 Query membership folder using binary file format 
- 2 Create membership view in bronze schema 



### 1 Query membership folder using binary file format 


In [0]:
%fs ls '/Volumes/gizmobox/landing/operational/memberships'